# 02 — Chunking + retrieval test

Goal: for each clause type, does the right chunk show up in the top-k search results? If recall@5 is low here, the LLM never sees the clause, so fix this before writing any prompts.

In [3]:
import sys, json
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

sys.path.insert(0, "../src")
from chunking import chunk_contract

d = json.load(open("../data/CUADv1.json"))["data"]
split = json.load(open("../data/split.json"))
dev_ids = set(split["dev"])
docs = {x["title"]: x for x in d if x["title"] in dev_ids}
print(len(docs), "dev contracts")

W1113 11:56:25.762000 98394 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


410 dev contracts


## Clause types and search queries
Queries are based on CUAD's own description of each clause type (`category_descriptions.csv` in the CUAD repo).

In [4]:
QUERIES = {
    "Termination For Convenience": "Either party may terminate this agreement at any time without cause upon written notice.",
    "Post-Termination Services": "Obligations that continue after termination or expiration, such as transition services, final payments, return of materials.",
    "Exclusivity": "Exclusive dealing: a party must buy or sell only through the other party, or is the exclusive distributor or supplier.",
    "Renewal Term": "This agreement shall automatically renew for successive renewal terms unless either party gives notice of non-renewal.",
    "Insurance": "A party shall maintain insurance coverage, such as general liability insurance, with minimum limits and name the other party as additional insured.",
    "Minimum Commitment": "Minimum purchase or order quantity, minimum amount or units that must be bought each year or period.",
    "Ip Ownership Assignment": "Intellectual property, work product and inventions created shall be owned by and assigned to the other party.",
    "Change Of Control": "If a party undergoes a change of control, merger, or sale of its assets, the other party may terminate or must consent.",
    "Non-Compete": "A party shall not compete with the other party or engage in a competing business in a territory or field.",
    "Uncapped Liability": "Limitations of liability shall not apply to breaches of confidentiality, indemnification, gross negligence or willful misconduct.",
}

In [5]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
q_emb = model.encode(list(QUERIES.values()), normalize_embeddings=True)
q_emb = dict(zip(QUERIES.keys(), q_emb))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Gold spans for the 10 clause types

In [6]:
gold = {}  # (contract, clause) -> list of (start, end)
for title, doc in docs.items():
    for qa in doc["paragraphs"][0]["qas"]:
        clause = qa["id"].split("__")[-1]
        if clause in QUERIES and not qa["is_impossible"]:
            gold[(title, clause)] = [(a["answer_start"], a["answer_start"] + len(a["text"])) for a in qa["answers"]]
print(len(gold), "contract-clause pairs with an answer")

1245 contract-clause pairs with an answer


## Retrieve and score
A hit means at least one of the top-k chunks overlaps a gold span. Start with 50 contracts; this cell is the slow one.

In [7]:
K_VALUES = [1, 3, 5, 10]
SAMPLE = sorted(docs)[:50]   # raise to all dev contracts once it works

def overlaps(c, spans):
    return any(c["start"] < e and s < c["end"] for s, e in spans)

rows = []
for title in SAMPLE:
    text = docs[title]["paragraphs"][0]["context"]
    chunks = chunk_contract(text)
    emb = model.encode([c["text"] for c in chunks], normalize_embeddings=True, batch_size=64)
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb.astype("float32"))

    for clause, q in q_emb.items():
        spans = gold.get((title, clause))
        if not spans:
            continue
        _, idx = index.search(q[None].astype("float32"), max(K_VALUES))
        ranked = [chunks[i] for i in idx[0] if i >= 0]
        row = {"contract": title, "clause": clause}
        for k in K_VALUES:
            row[f"r@{k}"] = any(overlaps(c, spans) for c in ranked[:k])
        rows.append(row)

res = pd.DataFrame(rows)
res.groupby("clause")[[f"r@{k}" for k in K_VALUES]].mean().round(2).sort_values("r@5")

,r@1,r@3,r@5,r@10
clause,,,,
Non-Compete,0.00,0.14,0.36,0.57
Exclusivity,0.20,0.50,0.65,0.70
Minimum Commitment,0.29,0.57,0.71,0.81
Ip Ownership Assignment,0.43,0.71,0.79,0.93
Change Of Control,0.67,0.78,0.89,0.89
Post-Termination Services,0.44,0.83,0.89,0.89
Uncapped Liability,0.75,0.83,0.92,1.00
Renewal Term,0.71,0.86,0.93,0.93
Insurance,0.84,0.95,0.95,0.95


In [8]:
res[[f"r@{k}" for k in K_VALUES]].mean().round(2)

r@1     0.48
r@3     0.69
r@5     0.80
r@10    0.86
dtype: float64

## Look at misses
Pick the worst clause type from the table above and read what retrieval returned instead.

In [10]:
CLAUSE = "Uncapped Liability"   # change to the worst one
miss = res[(res.clause == CLAUSE) & (~res["r@5"])].head(3)
for title in miss.contract:
    text = docs[title]["paragraphs"][0]["context"]
    for s, e in gold[(title, CLAUSE)]:
        print("GOLD:", text[s:e][:400])
        print()

GOLD: In no event will either party be liable to the other for special,       incidental, or indirect damages or for any consequential damages       (including lost profits or savings), even if they are informed of the       possibility; provided that this Section 10 does not apply to Customer's       failure to pay any amounts owing to IBM under this Agreement (including       amounts owing for Service

